# Phutball transfer suite (llm_transfer)

Does phutball training carry over? The same suite runs on the base model, the self-stop warm start, the v4 curriculum
best (r14) and the gated-GRPO best, and `llm_transfer.report` compares them item by item (paired z) per family and
difficulty dial. Families, ordered by distance from the training task: **bridge** (the validation win puzzles rewritten
without the board: same logic, new surface; eval only, never trained on), **pointer** chasing (1-8 hops, broken chains),
**path** finding (no-route cases), **boxes** (state tracking, REMOVE-heavy half), **ruler_vt** (variable tracking in
2k / 8k tokens of filler), **babilong** (qa2 / qa3 at 0k-8k, qa19 path finding), **cogniload** (20-100 statements),
**core** (code data / control / information-flow dependence, yes/no). The prediction to test: gains that grow with
hops / depth, most on REMOVE-heavy tracking, and fewer false "none" answers.
**Hard suite** (`suite_v2_hard.jsonl`, results `<name>_hard`): the base model saturated boxes / ruler_vt / pointer / path
in v1, so they are re-run with longer chains, more entities and distractors (boxes: 12 boxes, 32-96 operations, REMOVE-heavy
half 3:3:1; pointer 12-24 hops; path 6-12 on 48 towns; ruler_vt 16-32 hops with 3 decoy chains in 4k / 10k tokens).
Both suites are frozen on Drive on first build, so every model is scored on the same items.
**Runtime:** A100. v1: 1,155 items, ~45 min per model; hard: 390 items, ~15-20 min per model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, subprocess, time, urllib.request
OUT = '/content/drive/MyDrive/phutball/transfer'; os.makedirs(OUT, exist_ok=True)
MODELS = {                                                 # name -> LoRA adapter (None: the base model)
    'base': None,
    'warm': '/content/drive/MyDrive/phutball/grpo_pilot/self_stop/adapter/final',
    'r14': '/content/drive/MyDrive/phutball/curriculum_v4/r14/adapter/final',
    'grpo': '/content/drive/MyDrive/phutball/grpo_cost/p11/final',  # the gated-GRPO best (gate_state.json 'best'); its eval below
}
MAXLEN = 20000; CAP = 6144
GRPO_EVAL = 'v11'                                          # the validation eval of MODELS['grpo'] (pN -> vN)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!test -d phutball-jax || git clone -b expert-dagger https://github.com/echoname6/phutball-jax.git
%cd /content/phutball-jax
!git pull -q && git log --oneline -1
!pip -q install -U "trl[vllm]" peft datasets 2>&1 | tail -3
!pip -q install flash-linear-attention
!pip -q uninstall -y torchao
!python -c "import trl, vllm, peft, transformers; print('trl', trl.__version__, 'vllm', vllm.__version__, 'peft', peft.__version__, 'transformers', transformers.__version__)"

In [ ]:
%cd /content/phutball-jax
# the suites are FROZEN on Drive: built once, then every model (and every later session) runs the same items
import shutil
for suite, flag in (('suite_v1.jsonl', []), ('suite_v2_hard.jsonl', ['--hard'])):
    if os.path.exists(f'{OUT}/{suite}'):
        shutil.copy(f'{OUT}/{suite}', f'llm_transfer/data/{suite}'); print('frozen suite from Drive:', suite)
    elif os.path.exists(f'llm_transfer/data/{suite}'):                 # this session's build: keep it
        shutil.copy(f'llm_transfer/data/{suite}', f'{OUT}/{suite}'); print('saved this session\'s suite to Drive:', suite)
    else:
        subprocess.run(['python', '-m', 'llm_transfer.build', *flag, '--out', f'llm_transfer/data/{suite}'], check=True)
        shutil.copy(f'llm_transfer/data/{suite}', f'{OUT}/{suite}')
adapters = {k: v for k, v in MODELS.items() if v}
!pkill -f "vllm serve" ; sleep 5
log = open('/content/vllm_serve.log', 'w')
p = subprocess.Popen(['vllm', 'serve', 'Qwen/Qwen3.5-9B', '--max-model-len', str(MAXLEN), '--gpu-memory-utilization', '0.90',
                      '--port', '8000', '--enable-lora', '--max-lora-rank', '32', '--max-loras', str(len(adapters)),
                      '--lora-modules', *[f'{k}={v}' for k, v in adapters.items()]], stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try: urllib.request.urlopen('http://localhost:8000/health', timeout=5); print('server up'); break
    except Exception: time.sleep(5)
else: raise RuntimeError(open('/content/vllm_serve.log').read()[-2000:])

In [ ]:
for name in MODELS:                                        # v1 suite, then the hard suite (results <name>_hard)
    model = 'Qwen/Qwen3.5-9B' if MODELS[name] is None else name
    for suite, tag in (('suite_v1.jsonl', ''), ('suite_v2_hard.jsonl', '_hard')):
        if os.path.exists(f'{OUT}/{name}{tag}.summary.json'): print('done:', name + tag); continue
        !python -m llm_transfer.run --suite llm_transfer/data/{suite} --model {model} --name {name}{tag} --max-tokens {CAP} --concurrency 64 | tail -60
        !cp llm_transfer/results/{name}{tag}.jsonl llm_transfer/results/{name}{tag}.summary.json {OUT}/


In [ ]:
!mkdir -p llm_transfer/results && cp {OUT}/*.jsonl {OUT}/*.summary.json llm_transfer/results/
names = ' '.join(n for n in MODELS if os.path.exists(f'{OUT}/{n}.summary.json'))
hard = ' '.join(n + '_hard' for n in MODELS if os.path.exists(f'{OUT}/{n}_hard.summary.json'))
EV4 = '/content/drive/MyDrive/phutball/curriculum_v4/eval'; EVG = '/content/drive/MyDrive/phutball/grpo_cost/eval'
!python -m llm_transfer.report {names} --phutball r14={EV4}/v14 grpo={EVG}/{GRPO_EVAL}
!python -m llm_transfer.report {names} --key correct
!python -m llm_transfer.report {hard}
!python -m llm_transfer.report {hard} --key correct


In [ ]:
# optional: stop billing when everything above has finished (queue this cell last)
# from google.colab import runtime; runtime.unassign()